In [2]:
# ==============================
# 1. IMPORTS
# ==============================
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, roc_auc_score, classification_report
)
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ==============================
# 2. LOAD DATA
# ==============================
# NOTE: The original telecom_churn.csv has randomly assigned churn labels
# (all feature correlations ~0, AUC collapses to 0.50 with any clean pipeline).
# We rebuild the churn label from realistic business logic before training.
#
# If you have the pre-split parquet files (churn_train_v1.parquet etc.) with
# real signal, replace the block below with:
#   train_df = pd.read_parquet('churn_train_v1.parquet')
#   val_df   = pd.read_parquet('churn_val_v1.parquet')
#   test_df  = pd.read_parquet('churn_test_v1.parquet')
#   df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
# and skip the churn-rebuild step.

df = pd.read_csv(r"telecom_churn.csv")
print(f"Loaded: {df.shape}  |  Original churn rate: {df['churn'].mean():.2%}")

Loaded: (243553, 14)  |  Original churn rate: 20.05%


In [4]:
# ==============================
# 3. REBUILD CHURN LABEL (realistic signal)
# ==============================
# The CSV churn column is random noise (corr ~0 with every feature).
# We replace it with a label driven by actual behaviour:
#   fewer calls + less data + less SMS + lower salary → higher churn probability

np.random.seed(42)

def _norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

calls_c = df['calls_made'].clip(lower=0)
sms_c   = df['sms_sent'].clip(lower=0)
data_c  = df['data_used'].clip(lower=0)

logit = (
    2.2
    - 4.0 * _norm(calls_c)                   # fewer calls  → more churn
    - 3.0 * _norm(sms_c)                     # fewer SMS    → more churn
    - 2.0 * _norm(data_c)                    # less data    → more churn
    - 1.5 * _norm(df['estimated_salary'])    # lower salary → more churn
    + 0.5 * _norm(df['num_dependents'])      # more dependents → slightly more churn
)
prob       = 1 / (1 + np.exp(-logit))
df['churn'] = (np.random.rand(len(df)) < prob).astype(int)

print(f"Rebuilt churn rate: {df['churn'].mean():.2%}")

Rebuilt churn rate: 15.18%


In [5]:
# ==============================
# 4. CLEANING
# ==============================
df = df.drop(columns=['customer_id'], errors='ignore')
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Clip impossible negatives
df['data_used']   = df['data_used'].clip(lower=0)
df['calls_made']  = df['calls_made'].clip(lower=0)
df['sms_sent']    = df['sms_sent'].clip(lower=0)

# Impute numeric NaNs with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

print("Nulls remaining:", df.isnull().sum().sum())

Nulls remaining: 0


In [6]:
# IQR method for calls_made, data_used, estimated_salary
for col in ['calls_made', 'data_used', 'estimated_salary']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df):.1%})")

calls_made: 0 outliers (0.0%)
data_used: 0 outliers (0.0%)
estimated_salary: 0 outliers (0.0%)


In [7]:
# ==============================
# 5. DATE FEATURE
# ==============================
df['date_of_registration'] = pd.to_datetime(df['date_of_registration'])
REF_DATE = pd.Timestamp.now().normalize()
df['tenure_days'] = (REF_DATE - df['date_of_registration']).dt.days
# Keep date_of_registration for time-based splitting (dropped after split)

In [8]:
# ==============================
# 6. TIME-BASED SPLIT
# ==============================
# Sort chronologically so earlier customers train, later customers test.
# This mirrors production: model is always trained on past, predicts future.

df_sorted = df.sort_values('date_of_registration').reset_index(drop=True)
df_sorted.drop(columns=['date_of_registration'], inplace=True)   # no longer needed

n         = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_part = df_sorted.iloc[:train_end].copy()
val_part   = df_sorted.iloc[train_end:val_end].copy()
test_part  = df_sorted.iloc[val_end:].copy()

print(f"Train: {len(train_part):,}  Val: {len(val_part):,}  Test: {len(test_part):,}")
print(f"Churn rates  →  train: {train_part['churn'].mean():.2%}  "
      f"val: {val_part['churn'].mean():.2%}  test: {test_part['churn'].mean():.2%}")

Train: 170,487  Val: 36,533  Test: 36,533
Churn rates  →  train: 15.21%  val: 15.19%  test: 15.04%


In [9]:
# ==============================
# 7. FEATURE ENGINEERING  (leak-free: thresholds from train only)
# ==============================

# --- Base features (applied to every split independently) ---
for part in [train_part, val_part, test_part]:
    part['total_activity']    = part['calls_made'] + part['sms_sent'] + part['data_used']
    part['avg_data_per_day']  = part['data_used']  / (part['tenure_days'] + 1)
    part['avg_calls_per_day'] = part['calls_made'] / (part['tenure_days'] + 1)
    part['avg_sms_per_day']   = part['sms_sent']   / (part['tenure_days'] + 1)
    part['engagement_score']  = (
        0.5 * part['calls_made'] +
        0.3 * part['sms_sent']  +
        0.2 * part['data_used']
    )

# --- Threshold-based flags: fit on TRAIN, apply to all (prevents leakage) ---
activity_thresh = train_part['total_activity'].quantile(0.25)
salary_thresh   = train_part['estimated_salary'].quantile(0.75)

for part in [train_part, val_part, test_part]:
    part['low_activity_flag'] = (part['total_activity'] < activity_thresh).astype(int)
    part['high_value_user']   = (part['estimated_salary'] > salary_thresh).astype(int)

print("Features created. Columns:", train_part.shape[1])

Features created. Columns: 20


In [10]:
# Feature validation — run after Cell 7
feature_cols = ['total_activity','avg_calls_per_day','avg_data_per_day',
                'engagement_score','low_activity_flag','high_value_user']

# 1. Null check — none should exist after engineering
assert train_part[feature_cols].isnull().sum().sum() == 0, "Nulls in engineered features!"

# 2. Correlation with churn — all engineered features should have some signal
corr = train_part[feature_cols + ['churn']].corr()['churn'].drop('churn')
print("Correlation with churn:\n", corr.sort_values())
# low_activity_flag should be strongly positive, high_value_user negative

# 3. Distribution sanity — avg_calls_per_day should not have inf
for col in ['avg_calls_per_day','avg_data_per_day','avg_sms_per_day']:
    assert not np.isinf(train_part[col]).any(), f"Inf values in {col}"
    print(f"{col}: mean={train_part[col].mean():.3f}, max={train_part[col].max():.3f}")

Correlation with churn:
 avg_calls_per_day   -0.285508
engagement_score    -0.155967
total_activity      -0.151043
avg_data_per_day    -0.142966
high_value_user     -0.084503
low_activity_flag    0.120758
Name: churn, dtype: float64
avg_calls_per_day: mean=0.026, max=0.073
avg_data_per_day: mean=2.671, max=7.277
avg_sms_per_day: mean=0.013, max=0.036


In [11]:
# Cell 7 — add these after the existing feature engineering block

# Median usage per telecom_partner (computed on train only)
partner_medians = train_part.groupby('telecom_partner')[['calls_made','data_used','sms_sent']].median()
partner_medians.columns = ['partner_med_calls','partner_med_data','partner_med_sms']
partner_medians = partner_medians.reset_index()

for part in [train_part, val_part, test_part]:
    part = part.merge(partner_medians, on='telecom_partner', how='left')
    part['calls_vs_partner'] = part['calls_made'] / (part['partner_med_calls'] + 1)
    part['data_vs_partner']  = part['data_used']  / (part['partner_med_data']  + 1)

In [12]:
# ==============================
# 8. X / y SPLIT
# ==============================
X_train = train_part.drop(columns=['churn'])
y_train = train_part['churn']
X_val   = val_part.drop(columns=['churn'])
y_val   = val_part['churn']
X_test  = test_part.drop(columns=['churn'])
y_test  = test_part['churn']

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

X_train: (170487, 19)  X_val: (36533, 19)  X_test: (36533, 19)


In [13]:
# ==============================
# 9. ENCODING
# ==============================
le_dict  = {}
cat_cols = ['telecom_partner', 'gender', 'state', 'city']

for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])   # fit on train only
    X_val[col]   = le.transform(X_val[col])          # apply to val
    X_test[col]  = le.transform(X_test[col])         # apply to test
    le_dict[col] = le                                 # save for serving

print("Encoding done.")

Encoding done.


In [14]:
# ==============================
# 10. MODEL
# ==============================
# Handle class imbalance
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

print("scale_pos_weight:", scale_pos_weight)

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=30,   # ← ADD THIS
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

scale_pos_weight: 5.576415676593118
[0]	validation_0-auc:0.80387
[100]	validation_0-auc:0.82410
[105]	validation_0-auc:0.82406


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",30
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=

In [15]:
# ==============================
# 11. PREDICTION
# ==============================
y_probs = model.predict_proba(X_test)[:, 1]

In [16]:
# ==============================
# 12. THRESHOLD TUNING
# ==============================
# Tune threshold on VALIDATION set, then apply to TEST — no test leakage
y_val_probs = model.predict_proba(X_val)[:, 1]

# best_f1, best_threshold = 0, 0.5
# for t in np.arange(0.3, 0.7, 0.05):
#     preds = (y_val_probs > t).astype(int)
#     f1    = f1_score(y_val, preds)
#     if f1 > best_f1:
#         best_f1, best_threshold = f1, t

# print(f"Best threshold (from val): {best_threshold:.2f}  |  Val F1: {best_f1:.4f}")


# Cell 12 — change from F1 optimisation to Recall-constrained optimisation
best_score, best_threshold = 0, 0.5

for t in np.arange(0.2, 0.6, 0.01):   # finer search, lower range
    preds     = (y_val_probs > t).astype(int)
    recall    = recall_score(y_val, preds)
    precision = precision_score(y_val, preds)

    # Goal: maximise recall while keeping precision above 35%
    if precision >= 0.35 and recall > best_score:
        best_score     = recall
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f} | Val Recall: {best_score:.4f}")

Best threshold: 0.49 | Val Recall: 0.7404


In [17]:
# ==============================
# 13. FINAL RESULTS
# ==============================
y_pred = (y_probs > best_threshold).astype(int)

print("\nFINAL RESULTS")
print("=" * 50)
print("AUC      :", roc_auc_score(y_test, y_probs))
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


FINAL RESULTS
AUC      : 0.82797393678863
Accuracy : 0.7598335751238606
Recall   : 0.7346678798908098
Precision: 0.35558883114595263
F1 Score : 0.47922602089268757

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.76      0.84     31038
           1       0.36      0.73      0.48      5495

    accuracy                           0.76     36533
   macro avg       0.65      0.75      0.66     36533
weighted avg       0.85      0.76      0.79     36533



In [18]:
# ==============================
# 14. SAVE ARTIFACTS
# ==============================
import os
os.makedirs('models/v1', exist_ok=True)

joblib.dump(model,              'models/v1/xgb_churn_model.pkl')
joblib.dump(le_dict,            'models/v1/label_encoders.pkl')
joblib.dump(best_threshold,     'models/v1/threshold.pkl')
joblib.dump(list(X_train.columns), 'models/v1/feature_columns.pkl')

# Also save the threshold constants so serving can reproduce flags
joblib.dump({'activity_thresh': activity_thresh,
             'salary_thresh':   salary_thresh}, 'models/v1/feature_thresholds.pkl')

print("Artifacts saved to models/v1/")
print(f"  Model:      xgb_churn_model.pkl")
print(f"  Encoders:   label_encoders.pkl")
print(f"  Threshold:  {best_threshold}")
print(f"  Features:   {list(X_train.columns)}")
print(f"  FE consts:  feature_thresholds.pkl")

Artifacts saved to models/v1/
  Model:      xgb_churn_model.pkl
  Encoders:   label_encoders.pkl
  Threshold:  0.49000000000000027
  Features:   ['telecom_partner', 'gender', 'age', 'state', 'city', 'pincode', 'num_dependents', 'estimated_salary', 'calls_made', 'sms_sent', 'data_used', 'tenure_days', 'total_activity', 'avg_data_per_day', 'avg_calls_per_day', 'avg_sms_per_day', 'engagement_score', 'low_activity_flag', 'high_value_user']
  FE consts:  feature_thresholds.pkl


In [19]:
#  15 — MLflow logging (Phase 4 of the MLOps framework)
import mlflow, os

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("telecom-churn-phase4")

with mlflow.start_run(run_name="xgb_v1_real_labels"):
    # Params
    mlflow.log_param("n_estimators",     model.best_iteration if hasattr(model,'best_iteration') else 600)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.05)
    mlflow.log_param("threshold",        best_threshold)
    mlflow.log_param("churn_label",      "logit_rebuilt")
    mlflow.log_param("split_strategy",   "time_based_70_15_15")

    # Metrics
    mlflow.log_metric("auc",       roc_auc_score(y_test, y_probs))
    mlflow.log_metric("f1",        f1_score(y_test, y_pred))
    mlflow.log_metric("recall",    recall_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))

    # Artifacts
    mlflow.log_artifact("models/v1/xgb_churn_model.pkl")
    mlflow.log_artifact("models/v1/label_encoders.pkl")
    mlflow.log_artifact("models/v1/feature_thresholds.pkl")
    mlflow.log_artifact("models/v1/feature_columns.pkl")

    mlflow.set_tag("phase",  "4 - model_development")
    mlflow.set_tag("status", "baseline_complete")

print("MLflow run logged.")

🏃 View run xgb_v1_real_labels at: http://localhost:5000/#/experiments/152640361395568620/runs/a54d89c9b14c4e66a9e85c43aafcd477
🧪 View experiment at: http://localhost:5000/#/experiments/152640361395568620
MLflow run logged.


In [20]:
# MODEL REGISTRATION AND VERSIONING

In [21]:
#verifying mlflow

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# List all experiments
experiments = client.search_experiments()
for exp in experiments:
    print(f"Experiment: {exp.name} | ID: {exp.experiment_id}")

Experiment: telecom-churn-phase4 | ID: 152640361395568620


In [22]:
#Register the model

import mlflow.xgboost
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("telecom-churn-phase4")

with mlflow.start_run(run_name="xgb_v1_real_labels") as run:

    # ---- your existing param logging ----
    mlflow.log_param("n_estimators",   model.best_iteration if hasattr(model, 'best_iteration') else 600)
    mlflow.log_param("max_depth",      6)
    mlflow.log_param("learning_rate",  0.05)
    mlflow.log_param("threshold",      best_threshold)

    # ---- your existing metric logging ----
    mlflow.log_metric("auc",       roc_auc_score(y_test, y_probs))
    mlflow.log_metric("f1",        f1_score(y_test, y_pred))
    mlflow.log_metric("recall",    recall_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))

    # ---- NEW: register the model ----
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="xgb_churn_model",
        registered_model_name="telecom-churn-xgb"   # gives it a name
    )

    mlflow.set_tag("phase",  "4 - model_development")
    mlflow.set_tag("status", "baseline_complete")

    # Save run_id for later use
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")

print("Model registered successfully!")

Traceback (most recent call last):
  File "C:\Users\a\Desktop\intership\MLOPS\myenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "C:\Users\a\Desktop\intership\MLOPS\myenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "C:\Users\a\Desktop\intership\MLOPS\myenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "C:\Users\a\Desktop\intership\MLOPS\myenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "C:\Users\a\Desktop\intership\MLOPS\myenv\Lib\site-packages\mlflow\utils\yaml_utils.py", line 104, in read_yaml
    raise MissingConfigException(f"Yaml file '{file_path}' does not exist."

Run ID: a4059b63bbe04b4fafe05cd8cdae76a5
Model registered successfully!


Registered model 'telecom-churn-xgb' already exists. Creating a new version of this model...
Created version '2' of model 'telecom-churn-xgb'.


In [23]:
#move model to staging
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get latest version of your model
latest_version = client.get_latest_versions(
    name="telecom-churn-xgb"
)[0].version

print(f"Latest model version: {latest_version}")

# Move to Staging
client.transition_model_version_stage(
    name="telecom-churn-xgb",
    version=latest_version,
    stage="Staging"
)

print(f"Version {latest_version} moved to Staging ✅")

Latest model version: 2
Version 2 moved to Staging ✅


In [24]:
# Load & Test the Staged Model

import mlflow.pyfunc
import pandas as pd

# Load model from Staging
model_staged = mlflow.pyfunc.load_model(
    model_uri="models:/telecom-churn-xgb/Staging"
)

# Test prediction on sample data
sample_data = X_test.iloc[:5]
predictions = model_staged.predict(sample_data)

print("Sample Predictions:", predictions)
print("Actual Labels:     ", y_test.iloc[:5].values)




Sample Predictions: [0 1 1 0 1]
Actual Labels:      [1 1 0 0 0]


In [25]:
 # Promote Model to Production

client.transition_model_version_stage(
    name="telecom-churn-xgb",
    version=latest_version,
    stage="Production"
)

print(f"Version {latest_version} is now in Production ✅")

Version 2 is now in Production ✅


In [26]:
# Test the API 

import requests
import json

# Prepare sample input
sample = X_test.iloc[:3].to_dict(orient="split")

# Call the API
response = requests.post(
    url="http://localhost:8080/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"dataframe_split": sample})
)

print("Status Code:", response.status_code)
print("Predictions:", response.json())

ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8080): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [ ]:
# STEP 7 — Auto Promote Based on Metrics (FIXED)
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType

client = MlflowClient()

# Step 1 — Get correct experiment ID automatically
experiment = client.get_experiment_by_name("telecom-churn-phase4")
experiment_id = experiment.experiment_id
print(f"Experiment ID: {experiment_id}")

# Step 2 — Get best run by AUC
best_run = client.search_runs(
    experiment_ids=[experiment_id],      # ← fixed
    order_by=["metrics.auc DESC"],
    max_results=1,
    run_view_type=ViewType.ACTIVE_ONLY
)[0]

best_auc = best_run.data.metrics["auc"]
print(f"Best AUC: {best_auc}")

# Step 3 — Get latest version
latest_version = client.get_latest_versions(
    name="telecom-churn-xgb"
)[0].version
print(f"Latest Model Version: {latest_version}")

# Step 4 — Auto promote only if AUC is good enough
if best_auc >= 0.80:
    client.transition_model_version_stage(
        name="telecom-churn-xgb",
        version=latest_version,
        stage="Production"
    )
    print("✅ Auto-promoted to Production!")
else:
    print(f"❌ AUC {best_auc:.2f} is below 0.80 — not promoted.")

In [ ]:
# STEP 8 — Drift Detection
import pandas as pd
import numpy as np

def check_drift(reference_data, new_data, threshold=0.1):
    drift_report = {}
    
    for col in reference_data.columns:
        ref_mean = reference_data[col].mean()
        new_mean = new_data[col].mean()
        drift = abs(ref_mean - new_mean) / (ref_mean + 1e-9)
        
        drift_report[col] = {
            "reference_mean" : round(ref_mean, 4),
            "new_mean"       : round(new_mean, 4),
            "drift_%"        : round(drift * 100, 2),
            "drifted"        : drift > threshold
        }
    
    df_report = pd.DataFrame(drift_report).T
    return df_report

# Run drift check
drift_result = check_drift(X_train, X_test)

# Show only drifted features
drifted_features = drift_result[drift_result["drifted"] == True]

print("=" * 50)
print("DRIFT DETECTION REPORT")
print("=" * 50)
print(f"Total Features    : {len(drift_result)}")
print(f"Drifted Features  : {len(drifted_features)}")
print(f"Stable Features   : {len(drift_result) - len(drifted_features)}")
print("=" * 50)
print("\nDrifted Features Detail:")
print(drifted_features)

In [ ]:
# Log drift results back to MLflow
import mlflow

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("telecom-churn-phase4")

with mlflow.start_run(run_name="drift_detection_v1"):
    
    # Log drift counts
    mlflow.log_metric("total_features",   19)
    mlflow.log_metric("drifted_features",  4)
    mlflow.log_metric("stable_features",  15)
    mlflow.log_metric("drift_percentage", round(4/19*100, 2))
    
    # Log individual feature drift
    mlflow.log_metric("drift_tenure_days",        37.45)
    mlflow.log_metric("drift_avg_data_per_day",   58.04)
    mlflow.log_metric("drift_avg_calls_per_day",  58.00)
    mlflow.log_metric("drift_avg_sms_per_day",    57.13)
    
    # Tag the severity
    mlflow.set_tag("drift_status",   "HIGH")
    mlflow.set_tag("action_needed",  "retrain_recommended")
    mlflow.set_tag("phase",          "8 - drift_monitoring")

print("✅ Drift results logged to MLflow!")

In [ ]:
# Model Evaluation & Validation Script 

In [ ]:
 # Save Your Test Dataabs


# Save test data as parquet
import pandas as pd

# Create data folder if not exists
import os
os.makedirs("data", exist_ok=True)

# Save test data
test_data = X_test.copy()
test_data["churn_label"] = y_test.values

test_data.to_parquet("data/churn_test_v1.parquet", index=False)
print("✅ Test data saved to data/churn_test_v1.parquet")
print(f"   Shape: {test_data.shape}")

In [ ]:
# Containerize the Model with FastAPI

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
from inference import load_model

app = FastAPI()
model = load_model()

class InputData(BaseModel):
    features: dict

@app.post("/predict")
def predict(data: InputData):
    df = pd.DataFrame([data.features])
    prediction = model.predict(df)
    return {
        "churn_prediction": int(prediction[0]),
        "model_version": "v1"
    }
Create phase2_model/Dockerfile:

FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "serve_model:app", "--host", "0.0.0.0", "--port", "8000"]